In [ ]:
import os, glob, re, time, ollama
from ollama import Client

def process_images(image_dir, output_dir, merge_filename, model_name="llama3.2-vision:11b"):
    """
    處理指定資料夾內的圖片進行 OCR，並計算執行時間。
    """
    start_time = time.time()  # 記錄開始時間

    # 內部定義自然排序邏輯
    def natural_sort_key(s):
        return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    search_path = os.path.join(image_dir, "*.jpg")
    image_files = sorted(glob.glob(search_path), key=natural_sort_key)

    if not image_files:
        print(f"在 {image_dir} 中找不到任何 jpg 檔案。")
        return

    all_contents = []
    total_files = len(image_files)
    print(f"找到 {total_files} 張圖片，開始進行 OCR 處理...")

    for i, img_path in enumerate(image_files, 1):
        file_name = os.path.basename(img_path)
        base_name = os.path.splitext(file_name)[0]
        output_txt_path = os.path.join(output_dir, f"{base_name}.txt")

        # 記錄單張開始時間
        single_start = time.time()
        print(f"[{i}/{total_files}] 正在處理: {file_name}...", end=" ", flush=True)

        try:
            oollama = Client(host='http://localhost:11434', timeout=30000)
            response = oollama.chat(
                    model=model_name, 
                    messages=[{
                        'role': 'user',
                        'content': (
                            '### 任務目標\n'
                            '你是一個高精度的 OCR 引擎。請將圖片中的內容完整轉譯為純粹的 Markdown 格式。\n\n'
                            '### 嚴格規範\n'
                            '1. 語言：必須使用【繁體中文】，禁止出現簡體字。\n'
                            '2. 格式禁令：嚴禁輸出任何 HTML 標籤（例如：禁止出現 <br>、<p>、<div> 等）。\n'
                            '3. 換行處理：若需換行，請直接使用標準換行符，禁止使用 <br>。\n'
                            '4. 表格：偵測到表格時，必須使用 Markdown Table 語法，確保欄位與原始圖片一一對應。\n'
                            '5. 模糊處理：若字跡模糊無法辨識，請統一以 [?] 代替。\n'
                            '6. 禁止輸出：嚴禁任何開場白或結語，僅輸出 Markdown 內容。\n'
                            '7. 排版：保持與原圖一致的段落結構。'
                        ),
                        'images': [img_path]
                    }],
                    options={
                        'temperature': 0.1, # 降低隨機性，提高辨識穩定度
                    }
                )

            content = response['message']['content']

            with open(output_txt_path, 'w', encoding='utf-8') as f:
                f.write(content)

            header = f"\n\n{'='*20} PAGE: {file_name} {'='*20}\n\n"
            all_contents.append(header + content)
            
            # 計算單張耗時
            single_end = time.time()
            print(f"完成! (耗時: {single_end - single_start:.2f} 秒)")

        except Exception as e:
            print(f"\n處理 {file_name} 時發生錯誤: {e}")

    # 執行合併
    with open(merge_filename, 'w', encoding='utf-8') as f_merge:
        f_merge.write("\n".join(all_contents))
    
    # 總結統計
    end_time = time.time()
    total_duration = end_time - start_time
    avg_time = total_duration / total_files if total_files > 0 else 0
    print(f"\n" + "="*40)
    print(f"任務完成統計：")
    print(f"總處理圖片數：{total_files} 張")
    print(f"總共花費時間：{total_duration:.2f} 秒 (約 {total_duration/60:.2f} 分鐘)")
    print(f"平均每張耗時：{avg_time:.2f} 秒")
    print(f"合併檔案儲存於：{merge_filename}")
    print("="*40)

if __name__ == "__main__":
    process_images(image_dir='results', output_dir='test_results', merge_filename='merge_all.txt',
        model_name='qwen3-vl:30b')

找到 49 張圖片，開始進行 OCR 處理...
[1/49] 正在處理: page_1.jpg... 完成! (耗時: 2396.59 秒)
[2/49] 正在處理: page_2.jpg... 